# Проект спринта 27. Часть 2: эксперименты и обучение

Это вторая тетрадка проекта по извлечению именованных сущностей (NER).
Чтобы приступить к работе с ней, вы должны завершить работу с первой тетрадкой, `ydp_27p_eda.ipynb`, в которой вы исследовали данные и изучили логику токенизации и выравнивания меток.

В этой части проекта вы настроите оценку качества моделей в целом и на уровне целых сущностей, получите baseline, проведёте два эксперимента с разным объёмом обучающих данных, сравните модели, проанализируете ошибки и проверите итоговую модель на собственных текстах.

**Модель:** `bert-base-multilingual-cased`.

### Где выполнять эту часть проекта

Задания второй части требуют GPU, поэтому выполняйте их на виртуальной машине. Подключение к ней описано в уроке «Инструкция к виртуальной машине с GPU».

Помните, что время работы VM ограничено 10 часами и начинает расходоваться с момента её инициализации. Запускайте виртуальную машину, когда будете готовы обучать модели, и не забывайте отключать её, когда завершаете работу.

Брать заготовки из первой тетрадки не нужно: код в разделе «Подготовка данных» делает всё необходимое.

### Как работать с тетрадкой

Выполняйте ячейки последовательно. Дополняйте ячейки с пометкой `TODO`, сохраняйте
результаты вычислений и записывайте выводы в соответствующих ячейках Markdown.
Тестовую выборку не используйте для выбора настроек.

Структура тетрадки:

- Подготовка окружения и обеспечение воспроизводимости.
- Подготовка данных.
- Шаг 4 — подготовка к обучению и создание базовой модели.
- Шаг 5 — эксперимент на малой выборке.
- Шаг 6 — эксперимент на увеличенной выборке.
- Шаг 7 — сравнение результатов.
- Шаг 8 — анализ ошибок.
- Шаг 9 — сохранение модели и инференс.
- Шаг 10 — выводы и самопроверка по чек-листу.

## Подготовка окружения и обеспечение воспроизводимости

Установите зависимости, импортируйте библиотеки, зафиксируйте `SEED` и проверьте
доступное вычислительное устройство.

In [1]:
# Установка библиотек
%pip install -q numpy==2.2.6 pandas==3.0.1 matplotlib==3.10.8 torch==2.6.0 transformers datasets accelerate evaluate seqeval spacy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import evaluate
import matplotlib.pyplot as plt
from datasets import load_dataset
from seqeval.metrics.sequence_labeling import get_entities
from spacy import displacy
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
    set_seed,
)


In [3]:
SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")
print("Версии: " + ", ".join(
    f"{package}={version(package)}"
    for package in [
        "torch",
        "transformers",
        "datasets",
        "accelerate",
        "evaluate",
        "seqeval",
        "spacy",
    ]
))


Используемое устройство: cpu
Версии: torch=2.6.0+cpu, transformers=5.14.1, datasets=4.1.1, accelerate=1.15.0, evaluate=0.4.6, seqeval=1.2.2, spacy=3.8.7


## Подготовка данных для BERT

Этот блок повторяет подготовку из первой тетрадки: загружает WikiANN, собирает словари
меток, токенизирует предложения и выравнивает NER-метки с подтокенами BERT. Код приведён целиком — дополнять его не нужно.

Напомним политику выравнивания: первый токен слова получает исходную метку,
остальные токены, специальные токены и паддинг получают `-100` и исключаются
из расчёта функции потерь.

In [4]:
# Загрузка русскоязычной конфигурации WikiANN и словарей меток
raw_datasets = load_dataset("unimelb-nlp/wikiann", "ru")

label_list = raw_datasets["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}
ENTITY_TYPES = sorted({
    label.split("-")[-1]
    for label in label_list
    if label != "O"
})

print(f"Метки: {label_list}")
print(f"Типы сущностей: {ENTITY_TYPES}")

Метки: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Типы сущностей: ['LOC', 'ORG', 'PER']


In [5]:
model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


In [6]:
def tokenize_and_align_labels(examples):
    """Токенизирует слова и выравнивает NER-метки с подтокенами."""
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [7]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

# Эти переменные понадобятся для работы моделей
train_full = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["validation"]
test_dataset = tokenized_datasets["test"]

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Контрольная проверка: длины полей одного примера должны совпадать
example = train_full[0]
assert (
    len(example["input_ids"])
    == len(example["attention_mask"])
    == len(example["labels"])
)

print(f"Обучающая выборка:   {len(train_full)}")
print(f"Валидационная:       {len(eval_dataset)}")
print(f"Тестовая:            {len(test_dataset)}")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Обучающая выборка:   20000
Валидационная:       10000
Тестовая:            10000


## Подготовка к обучению

Ниже представлены несколько блоков кода, которые понадобятся вам для работы с моделями:

- код, который фиксирует общие параметры экспериментов;
- функция для инициализации свежей модели;
- функция для обучения модели.

Работая с моделями, используйте валидационный срез для выбора `learning_rate`, а полную тестовую выборку — только для итоговой оценки моделей. Между экспериментами меняется размер обучающей выборки, остальные настройки остаются одинаковыми.


In [8]:
# Настройки моделей
FEWSHOT_SIZE = 100
LARGE_SIZE = 1000
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
EVAL_SUBSET_SIZE = 1000
CANDIDATE_LEARNING_RATES = (5e-5, 2e-5)

eval_small = eval_dataset.shuffle(seed=SEED).select(range(EVAL_SUBSET_SIZE))
print(
    f"Валидация во время обучения: "
    f"{len(eval_small)} из {len(eval_dataset)} примеров"
)


Валидация во время обучения: 1000 из 10000 примеров


In [9]:
def init_model():
    """Создаёт свежую модель token classification с одинаковой инициализацией."""
    set_seed(SEED)
    return AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )


In [10]:
def train_model(train_dataset, lr, output_dir):
    """Создаёт свежую модель и обучает её с заданным learning rate."""
    model = init_model()
    steps_per_epoch = int(np.ceil(len(train_dataset) / TRAIN_BATCH_SIZE))

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        # Сохраняем чекпойнт каждой эпохи и в конце берём лучший по валидационному F1
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        learning_rate=lr,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        logging_steps=max(steps_per_epoch // 2, 1),
        seed=SEED,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_small,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer


all_results = {}

## Настройка метрик

Реализуйте преобразование логитов и меток в строковые IOB2-теги. Исключите позиции
`-100`. `compute_metrics` должен возвращать общие `precision`, `recall`, `f1`,
`accuracy`, а также `precision`, `recall`, `f1` по каждому типу сущности. Используйте
ключ `f1` для общего `f1` и ключи вида `PER_f1` для отдельных типов меток. Если тип
отсутствует в результате `seqeval`, верните для него `0.0`.


In [11]:
metric = evaluate.load("seqeval")


def align_predictions(predictions, labels):
    """Преобразует логиты и id-метки в строки, исключая позиции -100."""
    # TODO: выберите класс с максимальным логитом
    predictions = np.argmax(predictions, axis=2)
    all_pred = [] # список для предсказаний
    all_true = [] # список для истинных меток
    for pred, label in zip(predictions, labels):
        true_labels = [] # список для истинных меток одного текста
        true_predictions = [] # список для предсказаний одного текста
        for p, l in zip(pred, label):
            if l != -100: # исключаем позиции -100, не нужные для метрик
                true_labels.append(id2label[l]) # переводим id правильного класса в название тега
                true_predictions.append(id2label[p])
        all_pred.append(true_predictions) 
        all_true.append(true_labels)
        
    return all_pred, all_true      
    # TODO: удалите позиции -100 и преобразуйте id в label_list



def compute_metrics(p):
    """Возвращает общие метрики и метрики по PER, ORG, LOC."""
    predictions, labels = p # получаем логиты модели и правильные метки
    # TODO: вызовите align_predictions и metric.compute
    all_pred, all_true = align_predictions(predictions, labels) # приводим их к строковым NER-тегам и убираем позиции -100
    results = metric.compute( # считаем метрики seqeval
        predictions=all_pred,
        references=all_true
        )
    
    # TODO: соберите словарь общих метрик и метрик по ENTITY_TYPES
    metrics = {
        'precision': results["overall_precision"],
        'recall': results["overall_recall"],
        'f1': results["overall_f1"],
        'accuracy': results["overall_accuracy"]
    }
    for entity in ENTITY_TYPES:
        if entity in results:
            metrics[f"{entity}_precision"] = results[entity]["precision"]
            metrics[f"{entity}_recall"] = results[entity]["recall"]
            metrics[f"{entity}_f1"] = results[entity]["f1"]
        else:
            metrics[f"{entity}_precision"] = 0.0
            metrics[f"{entity}_recall"] = 0.0
            metrics[f"{entity}_f1"] = 0.0
    return metrics



In [12]:
def evaluate_model(trainer, dataset, name):
    """Выполняет один predict и возвращает метрики и строковые теги."""
    output = trainer.predict(dataset)
    true_predictions, true_labels = align_predictions(
        output.predictions,
        output.label_ids,
    )
    results = metric.compute(
        predictions=true_predictions,
        references=true_labels,
    )

    print(f"\n=== {name} ===")
    print(
        f"{'Overall':8} | "
        f"P: {results['overall_precision']:.4f} | "
        f"R: {results['overall_recall']:.4f} | "
        f"F1: {results['overall_f1']:.4f}"
    )
    for entity in ENTITY_TYPES:
        if entity in results:
            print(
                f"{entity:8} | "
                f"P: {results[entity]['precision']:.4f} | "
                f"R: {results[entity]['recall']:.4f} | "
                f"F1: {results[entity]['f1']:.4f}"
            )
        else:
            print(f"{entity:8} | не предсказан ни разу")

    return results, true_predictions, true_labels


## Шаг 4. Создание базовой модели

Baseline — предобученный BERT с новой необученной NER-головой. Не вызывайте
`train()` — сразу оцените свежую модель на полной тестовой выборке.


In [13]:
baseline_model = init_model()
baseline_trainer = Trainer(
    model=baseline_model,
    args=TrainingArguments(
        "bert-ner-baseline",
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        report_to="none",
    ),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

baseline_results, _, _ = evaluate_model(
    baseline_trainer,
    test_dataset,
    "Baseline без обучения",
)
all_results["baseline"] = baseline_results

del baseline_trainer, baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

d:\ЯПрактикум\ner-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params


=== Baseline без обучения ===
Overall  | P: 0.0218 | R: 0.0781 | F1: 0.0341
LOC      | P: 0.0574 | R: 0.1452 | F1: 0.0823
ORG      | P: 0.0234 | R: 0.0663 | F1: 0.0346
PER      | P: 0.0009 | R: 0.0054 | F1: 0.0016


### Выводы о базовой модели

> **TODO:** Запишите свои выводы после прогона baseline-модели.


### Выводы о базовой модели

Baseline-модель без дообучения показала очень низкое качество: общий F1 = 0.0341, precision = 0.0218, recall = 0.0781. Это ожидаемый результат, поскольку предобученный BERT ещё не был дообучен на задаче NER, а классификационный слой для предсказания классов сущностей был инициализирован заново.

Качество распознавания всех типов сущностей низкое. Лучше всего baseline распознаёт LOC (F1 = 0.0823), хуже — ORG (F1 = 0.0346), а практически не распознаёт PER (F1 = 0.0016).

Таким образом, одного предобученного представления BERT недостаточно для решения задачи NER без дополнительного обучения на размеченных примерах. Полученные результаты baseline можно использовать как отправную точку для оценки эффекта последующего дообучения.

> 💾 **Точка сохранения.** Первые метрики получены. Закоммитьте тетрадку, не откладывая дело в долгий ящик:
>
> ```bash
> git add . && git commit -m "Baseline без дообучения" && git push
> ```

## Шаг 5. Эксперимент на малой выборке

Выберите 100 воспроизводимых случайных примеров из полной обучающей выборки. Обучите две свежие
модели с `learning_rate=5e-5` и `learning_rate=2e-5`. Выберите лучший вариант по
F1 на одном и том же валидационном срезе.


In [14]:
# TODO: перемешайте train_full с SEED и выберите FEWSHOT_SIZE примеров
train_100 = train_full.shuffle(seed=SEED).select(range(FEWSHOT_SIZE))
len(train_100)

100

In [15]:
# TODO: обучите вариант A с первым learning_rate
lr_a = CANDIDATE_LEARNING_RATES[0]
trainer_lr_a = train_model(train_100, lr_a, "bert-ner-fewshot-lr-a")
val_f1_a = trainer_lr_a.evaluate(eval_small)["eval_f1"]

# TODO: выведите F1 варианта A на валидационном срезе
print(f"F1 variant A: {val_f1_a}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
1,1.288113,1.091813,0.488327,0.203239,0.287021,0.661307,0.000000,0.000000,0.000000,0.093023,0.031496,0.047059,0.620779,0.615979,0.618370
2,0.861538,0.800040,0.472661,0.314980,0.378037,0.727246,0.304183,0.171674,0.219479,0.162162,0.078740,0.106007,0.744000,0.719072,0.731324
3,0.556822,0.642394,0.444623,0.448583,0.446594,0.790087,0.364606,0.366953,0.365775,0.308123,0.288714,0.298103,0.650000,0.703608,0.675743
4,0.343442,0.576745,0.456422,0.483401,0.469524,0.807051,0.405242,0.431330,0.417879,0.356322,0.325459,0.340192,0.586207,0.701031,0.638498
5,0.349984,0.560259,0.455229,0.489879,0.471919,0.810592,0.401515,0.454936,0.426559,0.352601,0.320210,0.335626,0.595604,0.698454,0.642942


d:\ЯПрактикум\ner-project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
0.349984,0.560259,5,0.455229,0.489879,0.471919,0.810592,0.401515,0.454936,0.426559,0.352601,0.320210,0.335626,0.595604,0.698454,0.642942


F1 variant A: 0.4719188767550702


In [16]:
# TODO: обучите независимый вариант B со вторым learning_rate
lr_b = CANDIDATE_LEARNING_RATES[1]
trainer_lr_b = train_model(train_100, lr_b, "bert-ner-fewshot-lr-b")
val_f1_b = trainer_lr_b.evaluate(eval_small)["eval_f1"]

# TODO: выведите F1 варианта B на валидационном срезе
print(f"F1 variant B: {val_f1_b}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
1,1.383208,1.338921,0.000000,0.000000,0.000000,0.543886,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1.171027,1.081687,0.544757,0.172470,0.261993,0.648621,0.222222,0.004292,0.008421,0.098039,0.013123,0.023148,0.622356,0.530928,0.573018
3,0.920931,0.952772,0.491654,0.262348,0.342133,0.690957,0.308824,0.045064,0.078652,0.158140,0.089239,0.114094,0.715426,0.693299,0.704188
4,0.659868,0.877967,0.506438,0.286640,0.366081,0.706889,0.326923,0.072961,0.119298,0.201717,0.123360,0.153094,0.754144,0.703608,0.728000
5,0.709730,0.853556,0.502793,0.291498,0.369042,0.710724,0.301471,0.087983,0.136213,0.215247,0.125984,0.158940,0.759104,0.698454,0.727517


d:\ЯПрактикум\ner-project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\ЯПрактикум\ner-project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
0.709730,0.853556,5,0.502793,0.291498,0.369042,0.710724,0.301471,0.087983,0.136213,0.215247,0.125984,0.158940,0.759104,0.698454,0.727517


F1 variant B: 0.3690415171706817


In [ ]:
# TODO: выберите лучший learning_rate только по валидационному F1
best_lr = lr_a if val_f1_a >= val_f1_b else lr_b # лучшая скорость обучения
trainer_fewshot = trainer_lr_a if val_f1_a >= val_f1_b else trainer_lr_b # лучшая модель по F1 на валидации

# TODO: оцените выбранную модель на полном test_dataset
# Сохраните fewshot_results, pred_tags_fewshot и true_tags_test
# Добавьте fewshot_results в all_results
fewshot_results, pred_tags_fewshot, true_tags_test = evaluate_model(trainer_fewshot, test_dataset, "Few-shot обучение")
# Освободите модели, которые больше не нужны
all_results["fewshot"] = fewshot_results

del trainer_fewshot, trainer_lr_a, trainer_lr_b
if torch.cuda.is_available():
    torch.cuda.empty_cache()



=== Few-shot обучение ===
Overall  | P: 0.4232 | R: 0.4781 | F1: 0.4490
LOC      | P: 0.3800 | R: 0.4748 | F1: 0.4221
ORG      | P: 0.3172 | R: 0.2882 | F1: 0.3020
PER      | P: 0.5699 | R: 0.7008 | F1: 0.6286


: 

### Выводы о модели, дообученной на малой выборке

> **TODO:** укажите валидационный `f1` для двух значений `learning_rate`, обоснуйте выбор лучшей модели и определите, какой тип сущностей распознаётся хуже всего на тесте.


### Выводы о модели, дообученной на малой выборке

Для обучения на 100 примерах были проверены два значения learning_rate на одном и том же валидационном срезе:

learning_rate = 5e-5: F1 = 0.4719;

learning_rate = 2e-5: F1 = 0.3690.

Лучший результат на валидационной выборке показал вариант с learning_rate = 5e-5, поэтому он был выбран для дальнейших экспериментов.

На полной тестовой выборке выбранная модель получила общий F1 = 0.4490. Даже обучение всего на 100 примерах существенно улучшило результат по сравнению с baseline (F1 = 0.0341).

Качество по отдельным типам сущностей составило:

PER: F1 = 0.6286;

LOC: F1 = 0.4221;

ORG: F1 = 0.3020.

Хуже всего модель распознаёт организации (ORG). При этом сущности типа PER распознаются заметно лучше остальных. Результаты показывают, что даже небольшая размеченная выборка позволяет модели адаптироваться к задаче NER, однако качества на 100 примерах ещё недостаточно, особенно для распознавания организаций.

> 💾 **Точка сохранения.** Позади два обучения — самое время сохранить результат
> в репозиторий, чтобы не проводить их заново.

## Шаг 6. Эксперимент на увеличенной выборке

Выберите 1000 воспроизводимых случайных примеров из полной обучающей выборки.

Инициализируйте новую модель, используйте лучший `learning_rate` и не меняйте остальные настройки.


In [ ]:
# TODO: сформируйте train_1000 из train_full с тем же SEED
train_1000 = train_full.shuffle(seed=SEED).select(range(LARGE_SIZE))

# TODO: обучите свежую модель с best_lr и сохраните полученный объект
trainer_large = train_model(train_1000, best_lr, "bert-ner-large")
# TODO: оцените её на полном test_dataset
# Сохраните large_results, pred_tags_large и true_tags_large
large_results, pred_tags_large, true_tags_large = evaluate_model(trainer_large, test_dataset, "Large модель")
# Проверьте, что true_tags_large совпадают с true_tags_test
assert true_tags_large == true_tags_test, "true_tags_large не совпадают с true_tags_test"
# Добавьте large_results в all_results
all_results["large"] = large_results


> 💾 **Точка сохранения.** Это был самый долгий прогон модели в проекте! Закоммитьте тетрадку с посчитанными метриками прямо сейчас.

## Шаг 7. Сравнение результатов

Соберите в одной таблице данные по baseline и моделям, обученным на 100 и 1000 примерах.

Покажите общие `precision`, `recall`, `f1` и `f1` для `PER`, `ORG`, `LOC`.


In [ ]:
def results_to_row(results):
    """Преобразует результаты seqeval в строку сравнительной таблицы."""
    # TODO: добавьте overall precision, recall, F1 и F1 по ENTITY_TYPES
    row = {}
    row['precision'] = results['overall_precision']
    row['recall'] = results['overall_recall']
    row['f1'] = results['overall_f1']
    for entity in ENTITY_TYPES:
        row[f"{entity}_f1"] = results[entity]["f1"]
    return row

# TODO: сформируйте comparison_df из all_results и выведите таблицу
rows = {
    name: results_to_row(results)
    for name, results in all_results.items()
}

comparison_df = pd.DataFrame(rows).T

In [ ]:
# TODO: рассчитайте абсолютный и относительный прирост метрик
# при переходе от 100 к 1000 примерам
absolute_growth = comparison_df.loc["large"] - comparison_df.loc["fewshot"]
relative_growth = absolute_growth / comparison_df.loc["fewshot"] * 100
# TODO: определите тип сущности с наибольшим приростом F1
entity_f1_columns = [f"{entity}_f1" for entity in ENTITY_TYPES]
best_entity = absolute_growth[entity_f1_columns].idxmax()

print("Сравнение моделей:")
display(comparison_df)

print("Абсолютный прирост метрик при переходе от 100 к 1000 примерам:")
display(absolute_growth)

print("Относительный прирост метрик, %:")
display(relative_growth)

print("Наибольший абсолютный прирост F1 среди типов сущностей:", best_entity.replace("_f1", ""))

### Выводы о результатах сравнения моделей

> **TODO:** сравните baseline и две обученные модели. Укажите абсолютный и относительный прирост общего `f1`, изменение `precision` и `recall`, а также тип сущности с наибольшим приростом качества предсказания.


> 💾 **Точка сохранения.** Сравнительная таблица — ключевой результат проекта.
> Отправьте её в репозиторий.

## Шаг 8. Анализ ошибок

Проанализируйте ошибки на уровне целых сущностей. Используйте сохранённые предсказания, чтобы не запускать повторный `predict()` на полной тестовой выборке.

При анализе ориентируйтесь на типы ошибок из таблицы:

| Тип | Описание |
|---|---|
|Пропуск сущности|Сущность есть в эталоне, но отсутствует в предсказании.|
|Неверный тип|Границы совпали, но тип сущности отличается.|
|Неправильные границы|Предсказание пересекается с эталоном, но границы отличаются.|
|Лишняя сущность|Модель нашла сущность там, где её нет в эталоне.|


In [ ]:
test_tokens = raw_datasets["test"]["tokens"]

MISSED = "пропуск сущности"
WRONG_TYPE = "неверный тип"
WRONG_BOUNDARIES = "неправильные границы"
SPURIOUS = "лишняя сущность"
ERROR_KINDS = [MISSED, WRONG_TYPE, WRONG_BOUNDARIES, SPURIOUS]


def classify_errors(true_tags, pred_tags):
    """Сопоставляет сущности и классифицирует расхождения."""
    unmatched_pred = list(get_entities(pred_tags))
    errors = []

    for gold in get_entities(true_tags):
        gold_type, gold_start, gold_end = gold

        exact = next(
            (
                pred
                for pred in unmatched_pred
                if (pred[1], pred[2]) == (gold_start, gold_end)
            ),
            None,
        )
        if exact is not None:
            unmatched_pred.remove(exact)
            if exact[0] != gold_type:
                errors.append({
                    "kind": WRONG_TYPE,
                    "gold": gold,
                    "pred": exact,
                })
            continue

        overlapping = next(
            (
                pred
                for pred in unmatched_pred
                if pred[1] <= gold_end and gold_start <= pred[2]
            ),
            None,
        )
        if overlapping is not None:
            unmatched_pred.remove(overlapping)
            errors.append({
                "kind": WRONG_BOUNDARIES,
                "gold": gold,
                "pred": overlapping,
            })
            continue

        errors.append({"kind": MISSED, "gold": gold, "pred": None})

    for spurious in unmatched_pred:
        errors.append({
            "kind": SPURIOUS,
            "gold": None,
            "pred": spurious,
        })

    return errors


def format_entity(entity, tokens):
    """Возвращает сущность в виде «текст» [ТИП]."""
    if entity is None:
        return "—"
    entity_type, start, end = entity
    return f"«{' '.join(tokens[start:end + 1])}» [{entity_type}]"


### Поиск примеров ошибок

Выведите не менее пяти предложений, в которых ошибается модель, обученная на 1000 примеров. Для каждого покажите текст, эталонные и предсказанные теги, тип ошибки и сущности.


In [ ]:
NUM_ERROR_EXAMPLES = 5

# TODO: пройдите по true_tags_test и pred_tags_large
# TODO: выведите первые пять примеров, где classify_errors вернул ошибки
counter = 0 # счетчик найденых предложений с ошибками
for tokens, true_tags, pred_tags in zip(
    test_tokens,
    true_tags_test,
    pred_tags_large,
):
    errors = classify_errors(true_tags, pred_tags)
    if errors:
        print("Текст:", " ".join(tokens))
        print("Эталонные теги:", true_tags)
        print("Предсказанные теги:", pred_tags)
        for error in errors:
            print(
                f"{error['kind']}: "
                f"gold={format_entity(error['gold'], tokens)}, "
                f"pred={format_entity(error['pred'], tokens)}"
            )
        counter += 1
        if counter >= NUM_ERROR_EXAMPLES:
            break

### Сравнение частот ошибок

Посчитайте типы ошибок по всей тестовой выборке для обеих обученных моделей.
Сравните абсолютные значения и доли, затем определите самый частый тип ошибки.


In [ ]:
def count_error_kinds(true_tags_list, pred_tags_list):
    """Считает частоты типов ошибок по всему набору."""
    counter = Counter()
    for true_tags, pred_tags in zip(true_tags_list, pred_tags_list):
        for error in classify_errors(true_tags, pred_tags):
            counter[error["kind"]] += 1
    return counter

# TODO: постройте таблицу частот и долей ошибок для двух моделей
# TODO: выведите самый частый тип ошибки модели на 1000 примерах
errors_100 = count_error_kinds(true_tags_test, pred_tags_fewshot)
errors_1000 = count_error_kinds(true_tags_test, pred_tags_large)

# Общий объем ошибок
total_100 = sum(errors_100.values())
total_1000 = sum(errors_1000.values())

rows = [] # список словарей с частотами и долями ошибок
for kind in ERROR_KINDS:
    rows.append({
        "kind": kind,
        "count_100": errors_100[kind],
        "share_100": errors_100[kind] / total_100 * 100 if total_100 > 0 else 0,
        "count_1000": errors_1000[kind],
        "share_1000": errors_1000[kind] / total_1000 * 100 if total_1000 > 0 else 0,
    })
error_df = pd.DataFrame(rows)
# Самый частый тип ошибки модели на 1000 примерах
most_common_error_1000 = errors_1000.most_common(1)
print(f"Самый частый тип ошибки модели на 1000 примерах: {most_common_error_1000[0][0]}")

### Поиск расхождений между моделями

Найдите примеры, в которых каждая из обученных моделей показывает себя лучше другой. Для этого продемонстрируйте случаи, где модель на малой выборке права, а модель на увеличенной ошибается, и наоборот. Сравнивайте извлечённые сущности, а не только последовательности тегов.


In [ ]:
def same_entities(tags_a, tags_b):
    return get_entities(tags_a) == get_entities(tags_b)


def find_disagreements(true_tags_list, tags_a, tags_b, max_examples=2):
    """Возвращает индексы примеров, где одна модель права, а другая ошибается."""
    a_better = []
    b_better = []

    for i, true_tags in enumerate(true_tags_list):
        a_ok = same_entities(tags_a[i], true_tags)
        b_ok = same_entities(tags_b[i], true_tags)

        if a_ok and not b_ok and len(a_better) < max_examples:
            a_better.append(i)
        if b_ok and not a_ok and len(b_better) < max_examples:
            b_better.append(i)
        if len(a_better) >= max_examples and len(b_better) >= max_examples:
            break

    return a_better, b_better


# TODO: вызовите find_disagreements для двух обученных моделей
# TODO: выведите текст, эталон и оба предсказания для найденных примеров


### Выводы из анализа ошибок

> **TODO:** укажите самый частый тип ошибки, сравните общее число ошибок двух моделей, приведите примеры расхождений и сформулируйте ограничения модели.
> Учитывайте возможный шум автоматической разметки WikiANN.


> 💾 **Точка сохранения.** Анализ ошибок готов — время сделать коммит.

## Шаг 9. Сохранение модели и проверка на новых текстах

Используйте итоговую модель из эксперимента на увеличенной выборке. Получите
предсказания на небольшой фиксированной части тестовой выборки, сохраните модель
и токенизатор, загрузите их обратно и повторите предсказание на тех же примерах.
Результаты до и после загрузки должны полностью совпасть.


In [ ]:
# TODO: сохраните и повторно загрузите итоговую модель.
#
# 1. Получите предсказания на небольшой части test_dataset.
# 2. Сохраните модель через save_model(...) или save_pretrained(...),
# 3. Загрузите модель через AutoModelForTokenClassification.from_pretrained(...).
# 4. Повторите предсказание и проверьте совпадение через np.array_equal(...).


### Проверка модели на собственных текстах

Создайте 3–5 предложений, которых нет в WikiANN. Желательно включить однословные
и многословные сущности `PER`, `ORG`, `LOC`, а также сложные или неоднозначные примеры.


In [ ]:
# TODO: инициализируйте ner_pipeline через pipeline(...).
# Передайте модель, загруженную на предыдущем шаге, tokenizer,
# aggregation_strategy="first" и доступное устройство.


def visualize_predictions(text, pipeline_output):
    """Визуализирует предсказанные сущности через displacy."""
    ents = [
        {
            "start": entity["start"],
            "end": entity["end"],
            "label": entity["entity_group"],
        }
        for entity in pipeline_output
    ]
    displacy.render(
        {"text": text, "ents": ents, "title": None},
        style="ent",
        manual=True,
        jupyter=True,
    )


In [ ]:
custom_sentences = [
    # TODO: добавьте 3–5 собственных предложений
]

# TODO: Прогоните их через pipeline и визуализируйте через visualize_predictions

> 💾 **Точка сохранения.** Модель сохранена и проверена на своих текстах.
> Закоммитьте тетрадку: веса в репозиторий не попадут, они исключены в `.gitignore`.

## Шаг 10. Финальные выводы

> **TODO:** сформулируйте итог проекта своими словами. Проанализируйте:
>
> - выбранный `learning_rate` и основание его выбора;
> - отличие baseline от дообученных моделей;
> - как увеличение обучающей выборки повлияло на качество;
> - изменение качества по `PER`, `ORG`, `LOC`;
> - основные типы ошибок и ограничения;
> - результат проверки сохранения и инференс.

Отдельный блок посвятите рекомендациям заказчику. Объясните менеджерам «Эхолота», какие объёмы данных для разметки планировать и на какие детали обращать внимание при обучении. Отдельно проговорите нюансы, связанные с тем, что вы экспериментировали с энциклопедическими данными, а не с теми, которые обычно обрабатывает сервис медиамониторинга.



## Ресурсы

- [Hugging Face: Token Classification](https://huggingface.co/docs/transformers/tasks/token_classification)
- [Hugging Face Datasets](https://huggingface.co/docs/datasets/)
- [seqeval](https://github.com/chakki-works/seqeval)
- [WikiANN](https://huggingface.co/datasets/unimelb-nlp/wikiann)
- [bert-base-multilingual-cased](https://huggingface.co/bert-base-multilingual-cased)


## Чек-лист готовности второй части проекта

### Эксперименты

- [ ] Настройки экспериментов собраны в одном месте, `SEED` зафиксирован.
- [ ] Валидационный срез используется для выбора настроек, тестовая выборка — только для итоговой оценки.
- [ ] Каждый запуск начинается с новой модели с корректными `num_labels`, `id2label` и `label2id`.
- [ ] Метрики считаются через `seqeval` на уровне сущностей `PER`, `ORG` и `LOC`, позиции `-100` исключены.
- [ ] Получен baseline без вызова `train()`.
- [ ] На 100 примерах проверены два значения `learning_rate`, лучший вариант выбран по валидационному `F1`.
- [ ] Модель на 1000 примеров обучена с выбранным `learning_rate` и теми же остальными настройками.
- [ ] Обе обученные модели оценены на одной полной тестовой выборке.
- [ ] Все три варианта модели представлены в одной сравнительной таблице.
- [ ] Рассчитан абсолютный и относительный прирост качества после увеличения выборки.
- [ ] Сформулированы выводы о роли дообучения и влиянии объёма обучающих данных.

### Анализ и итог

- [ ] Показано минимум пять ошибок модели на увеличенной выборке.
- [ ] Посчитаны частоты типов ошибок.
- [ ] Найдены примеры, в которых каждая из двух обученных моделей превосходит другую.
- [ ] Лучшая модель сохранена и повторно загружена, предсказания совпадают.
- [ ] Выполнен инференс на 3–5 собственных предложениях с демонстрацией результатов.
- [ ] Написаны выводы по каждому блоку и финальный вывод, в котором зафиксированы конкретные значения и ограничения.
- [ ] Предоставлены рекомендации компании «Эхолот».
- [ ] Описаны нюансы, связанные с отличиями текстов Википедии и «Эхолота».

### Формальные требования

- [ ] Во всех случайных операциях используется единый `SEED`.
- [ ] Все обязательные `TODO` заполнены, временные заглушки удалены.
- [ ] Ядро перезапущено, тетрадка выполнена сверху вниз без ошибок.
- [ ] Результаты вычислений, таблицы и визуализации сохранены в тетрадке.
- [ ] Веса моделей не добавлены в репозиторий.